# 03. Station context: experiments

`03context1.ipynb` found only weak links between a station's surroundings and its weekend ratio. This notebook runs two experiments to find out why.

| Section | Experiment | Question |
|---|---|---|
| 1 | Well-mapped stations only | Is the weak result caused by gaps in OpenStreetMap? |
| 2 | Building floor area | Do office and residential **buildings** measure the work and home ends of commuting better than office tags and land use? |
| 3 | Summary | Which measures matter, and are they worth putting on the map? |

**Reading the numbers:** each check is a rank correlation with the weekend ratio, from -1 to +1. Negative means "more of this goes with weekday commuting"; positive means "more of this goes with weekend activity". As a rough guide, below 0.2 either way is weak, 0.2 to 0.4 moderate, above 0.4 strong.

Needs the outputs of `01datapreparation.ipynb` and `03context1.ipynb`.

## 0. Settings

In [ ]:
from pathlib import Path
import json
from datetime import datetime
import pandas as pd
import geopandas as gpd
import osmnx as ox

MIN_MAPPED_SHARE = 0.6    # <- Experiment 1: keep stations whose circle is at least this share mapped (0.6 = 60%)
REFRESH_OSM = False       # <- True = fetch the buildings again

# Experiment 2: which OpenStreetMap building types count as each end of a commute
WORK_BUILDINGS = ["office", "commercial"] # <- work end
HOME_BUILDINGS = ["apartments", "residential", "house", "terrace", "detached", "semidetached_house"] # <- home end
DEFAULT_LEVELS = 1        # <- storeys assumed when a building's height is not mapped

PROCESSED = Path("data/processed")
RAW = Path("data/raw")
METRIC_CRS = "EPSG:32647"

radius = json.loads((PROCESSED / "context_metadata.json").read_text())["radius_m"]
BUILDINGS_CACHE = RAW / f"osm_buildings_{radius}m.parquet"

stations = gpd.read_file(PROCESSED / "stations.geojson")
context = pd.read_csv(PROCESSED / "station_context.csv")
data = stations[["key", "name", "district", "weekend_ratio", "avg_daily_entries", "geometry"]].merge(context, on="key")
print(f"{len(data)} stations, radius {radius} m")

def rank_corr(df, cols, target="weekend_ratio"):
    """Spearman rank correlation of each column with the target, using pandas only."""
    return df[cols].rank().corrwith(df[target].rank()).round(2)

154 stations, radius 500 m


## 1. Experiment 1: well-mapped stations only

`share_mapped` is the share of each station's circle covered by any mapped land use. If gaps in OpenStreetMap were hiding the pattern, the correlations should get stronger when only well-mapped stations are kept.

In [8]:
place_cols = ["shops", "food", "offices", "education", "healthcare", "tourism_leisure"]
share_cols = ["share_residential", "share_commercial", "share_industrial", "share_green"]
cols = place_cols + share_cols

well = data[data["share_mapped"] >= MIN_MAPPED_SHARE]
print(f"Well-mapped stations: {len(well)} of {len(data)} (at least {MIN_MAPPED_SHARE:.0%} of the circle mapped)")

exp1 = pd.DataFrame({"all stations": rank_corr(data, cols), "well-mapped only": rank_corr(well, cols)})
exp1["change"] = (exp1["well-mapped only"].abs() - exp1["all stations"].abs()).round(2)
display(exp1.sort_values("well-mapped only"))

print("Where the well-mapped stations are:")
print(well["district"].value_counts().to_string())

Well-mapped stations: 44 of 154 (at least 60% of the circle mapped)


,all stations,well-mapped only,change
share_green,-0.06,-0.22,0.16
share_residential,-0.18,-0.13,-0.05
education,0.03,-0.09,0.06
tourism_leisure,0.04,0.03,-0.01
share_industrial,-0.04,0.07,0.03
offices,-0.00,0.10,0.10
food,0.11,0.21,0.10
share_commercial,0.16,0.23,0.07
healthcare,0.17,0.25,0.08
shops,0.15,0.30,0.15


Where the well-mapped stations are:
district
Petaling             21
W.P. Kuala Lumpur    16
Ulu Langat            3
Klang                 2
Sepang                1
W.P. Putrajaya        1


## 2. Experiment 2: building floor area

Office towers and apartment blocks are often mapped as **buildings** (e.g. `building=office`, `building=apartments`) without other tags, so 03context1 missed them. Here each building's floor area is estimated as its footprint times its number of storeys (`building:levels`), using `DEFAULT_LEVELS` where the height is not mapped. A building counts for a station if its centre lies inside the station's circle.

The first run fetches buildings from OpenStreetMap and can take several minutes.

In [9]:
circles = data[["key", "geometry"]].to_crs(METRIC_CRS)
circles["geometry"] = circles.geometry.buffer(radius)
search_area = circles.to_crs("EPSG:4326").union_all()

if REFRESH_OSM or not BUILDINGS_CACHE.exists():
    print("Fetching buildings from OpenStreetMap (this can take several minutes)...")
    b = ox.features_from_polygon(search_area, tags={"building": WORK_BUILDINGS + HOME_BUILDINGS})
    b = b[b.geometry.geom_type.isin(["Polygon", "MultiPolygon"])].reset_index()
    if "building:levels" not in b.columns:
        b["building:levels"] = None
    b = b[["building", "building:levels", "geometry"]].copy()
    b["building"] = b["building"].astype("string")
    b["building:levels"] = b["building:levels"].astype("string")
    b["fetched_on"] = datetime.now().strftime("%Y-%m-%d")
    b.to_parquet(BUILDINGS_CACHE)
else:
    print(f"Using saved copy {BUILDINGS_CACHE}")
buildings = gpd.read_parquet(BUILDINGS_CACHE).to_crs(METRIC_CRS)
print(f"{len(buildings):,} buildings, fetched on {buildings['fetched_on'].iloc[0]}")
print(buildings["building"].value_counts().to_string())

Using saved copy data\raw\osm_buildings_500m.parquet
69,222 buildings, fetched on 2026-09-30
building
house                 54642
commercial             5821
terrace                3251
semidetached_house     2342
apartments             1986
residential             704
detached                305
office                  171


In [10]:
# Floor area = footprint x storeys
levels = pd.to_numeric(buildings["building:levels"].str.extract(r"(\d+\.?\d*)")[0], errors="coerce")
print(f"Buildings with storeys mapped: {levels.notna().mean():.0%}")
buildings["footprint_m2"] = buildings.geometry.area
buildings["floor_m2"] = buildings["footprint_m2"] * levels.fillna(DEFAULT_LEVELS).clip(lower=1)
buildings["end"] = buildings["building"].map(lambda t: "work" if t in WORK_BUILDINGS else "home")

centres = buildings[["end", "floor_m2", "geometry"]].copy()
centres["geometry"] = buildings.geometry.representative_point()
joined = gpd.sjoin(centres, circles, predicate="within")

floor = joined.pivot_table(index="key", columns="end", values="floor_m2", aggfunc="sum", fill_value=0)
floor = floor.reindex(columns=["work", "home"], fill_value=0).rename(
    columns={"work": "work_floor_m2", "home": "home_floor_m2"}).round(0)
floor["commute_floor_m2"] = floor["work_floor_m2"] + floor["home_floor_m2"]
floor["work_building_count"] = joined[joined["end"] == "work"].groupby("key").size()
floor = floor.fillna(0)

data2 = data.merge(floor, left_on="key", right_index=True, how="left").fillna({c: 0 for c in floor.columns})

print("\nMost work floor area within the circle:")
display(data2.nlargest(8, "work_floor_m2")[["name", "weekend_ratio", "work_floor_m2", "work_building_count", "offices"]])

Buildings with storeys mapped: 7%

Most work floor area within the circle:


,name,weekend_ratio,work_floor_m2,work_building_count,offices
145,Tun Razak Exchange,0.842,733932.0,7.0,6
97,Raja Chulan,0.336,515269.0,14.0,19
65,KL Sentral,0.692,498631.0,13.0,25
5,Ampang Park,0.415,455761.0,9.0,26
64,KLCC,0.663,422478.0,10.0,36
85,Persiaran KLCC,0.536,415478.0,9.0,26
60,Kerinchi,0.266,411929.0,3.0,98
148,Universiti,0.491,299442.0,1.0,6


In [11]:
new_cols = ["work_floor_m2", "home_floor_m2", "commute_floor_m2", "offices", "shops", "food"]
well2 = data2[data2["share_mapped"] >= MIN_MAPPED_SHARE]
exp2 = pd.DataFrame({"all stations": rank_corr(data2, new_cols), "well-mapped only": rank_corr(well2, new_cols)})
display(exp2)
print("Expectation if the idea holds: work, home and commute floor area negative (weekday commuting);")
print("shops and food positive (weekend activity). 'offices' is the old tag-based measure, for comparison.")

,all stations,well-mapped only
work_floor_m2,-0.14,-0.07
home_floor_m2,0.00,0.08
commute_floor_m2,-0.05,0.09
offices,-0.00,0.10
shops,0.15,0.30
food,0.11,0.21


Expectation if the idea holds: work, home and commute floor area negative (weekday commuting);
shops and food positive (weekend activity). 'offices' is the old tag-based measure, for comparison.


## 3. Summary

The table below collects the strongest measures from both experiments. Measures with at least a moderate link (0.2 or more either way) are worth showing on the map; weak ones can still be shown as description, but should not be presented as explaining the weekend ratio.

In [12]:
summary = pd.concat([exp1["well-mapped only"], exp2["well-mapped only"]]).groupby(level=0).first()
summary = summary.reindex(summary.abs().sort_values(ascending=False).index)
strength = pd.cut(summary.abs(), [-0.01, 0.2, 0.4, 1], labels=["weak", "moderate", "strong"])
display(pd.DataFrame({"rank correlation (well-mapped)": summary, "strength": strength}))

# Save the building measures for the map
floor.reset_index().rename(columns={"index": "key"}).to_csv(PROCESSED / "station_buildings.csv", index=False)
print(f"Saved station_buildings.csv ({(PROCESSED / 'station_buildings.csv').stat().st_size / 1e3:.0f} KB)")

,rank correlation (well-mapped),strength
shops,0.30,moderate
healthcare,0.25,moderate
share_commercial,0.23,moderate
share_green,-0.22,moderate
food,0.21,moderate
share_residential,-0.13,weak
offices,0.10,weak
commute_floor_m2,0.09,weak
education,-0.09,weak
home_floor_m2,0.08,weak


Saved station_buildings.csv (6 KB)


___
**END**